# Project 4 — Notebook 3: RAG Pipeline

Game of Thrones Lore RAG System

This notebook implements a retrieval-augmented generation (RAG) question-answering system over the Game of Thrones lore database. The pipeline takes a natural language query, retrieves relevant sentences from Elasticsearch using hybrid search, constructs a prompt, and gets an answer from a local language model via Ollama.

## Imports

In [12]:
import os
from elasticsearch import Elasticsearch, helpers
import urllib3
from pprint import pprint

# Disable the specific InsecureRequestWarning
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [13]:
from sentence_transformers import SentenceTransformer

In [14]:
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Connect to Elasticsearch

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

ES_USER = os.getenv('ES_USER', 'elastic')
ES_PASSWORD = os.environ['ES_PASSWORD']  # set in .env
ES_HOST = os.getenv('ES_HOST', 'https://localhost:9200/')

client = Elasticsearch(
    ES_HOST,
    basic_auth=(ES_USER, ES_PASSWORD),
    verify_certs=False,
)

In [16]:
client.info()

ObjectApiResponse({'name': 'f08d4b694cd1', 'cluster_name': 'docker-cluster', 'cluster_uuid': 'sHThI9v4QFK889Tu6Z4A8w', 'version': {'number': '9.3.1', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': '0dd66e52ba3aa076cf498264e46339dbb71f0269', 'build_date': '2026-02-23T23:37:38.684779921Z', 'build_snapshot': False, 'lucene_version': '10.3.2', 'minimum_wire_compatibility_version': '8.19.0', 'minimum_index_compatibility_version': '8.0.0'}, 'tagline': 'You Know, for Search'})

## Setup LLM

We use Ollama running locally with the `llama3.2` model, which produces much higher quality answers than `tinyllama`.

In [17]:
from ollama import chat
from ollama import ChatResponse

In [18]:
response: ChatResponse = chat(model='llama3.2', messages=[
  {
    'role': 'user',
    'content': 'In one sentence, what is Game of Thrones about?',
  },
])
print(response['message']['content'])

Game of Thrones is a fantasy drama series based on George R.R. Martin's book series, following the battle for control of the Seven Kingdoms of Westeros as various noble families vie for the Iron Throne.


## Implement the RAG Pipeline

[1] Write a function to retrieve sentences from the database using hybrid search (bool with match and KNN should clauses).

In [19]:
def retrieve_sentences(query, count=10, max_length=512):
    query_vector = model.encode(query)

    search_query = {'query': {
        'bool': {
            'should': [
                {'match': {'sentence': query}},
                {'knn': {
                    'field': 'embedding',
                    'query_vector': query_vector,
                    'k': count,
                    'num_candidates': 5 * count
                }}
            ]
        }
    },
    'size': 3 * count
    }

    results_obj = client.search(index='got_lore', body=search_query)
    results = dict(results_obj)

    seen = set()
    sentences = []
    for hit in results['hits']['hits']:
        text = hit['_source']['sentence']
        if text not in seen:
            seen.add(text)
            sentences.append(text[:max_length])
        if len(sentences) == count:
            break

    return sentences

[2] Test the retrieval function.

In [20]:
test_sentences = retrieve_sentences('What happened at the Red Wedding?')
test_sentences

['She is married to Lord Edmure Tully as compensation at what becomes known as the Red Wedding.',
 'However, just as they arrive, the Red Wedding happens and the Freys begin slaughtering the Starks.',
 'Salon.com\'s Andrew Leonard "couldn\'t stop reading Martin because my desire to know what was going to happen combined with my absolute inability to guess what would happen and left me helpless before his sorcery.',
 'A long prologue was to establish what had happened in the meantime, initially just as one chapter of Aeron Damphair on the Iron Islands at the Kingsmoot.',
 'Robb Stark is slaughtered at the Red Wedding soon after, while Joffrey is poisoned at his own wedding to Margaery Tyrell.',
 'The wedding is a trap, with Robb, his key supporters and most of his army massacred during the feast, a direct violation of ancient guest right customs, in what becomes known as the Red Wedding.',
 'Robb is also slain at The Red Wedding along with his mother.',
 'At the Red Wedding, Roose betra

[3] Write a function to build the context string from retrieved sentences.

In [21]:
def create_context(sentences):
    return 'CONTEXT:\n * ' + '\n * '.join(sentences) + '\n'

In [22]:
context_str = create_context(test_sentences)
print(context_str)

CONTEXT:
 * She is married to Lord Edmure Tully as compensation at what becomes known as the Red Wedding.
 * However, just as they arrive, the Red Wedding happens and the Freys begin slaughtering the Starks.
 * Salon.com's Andrew Leonard "couldn't stop reading Martin because my desire to know what was going to happen combined with my absolute inability to guess what would happen and left me helpless before his sorcery.
 * A long prologue was to establish what had happened in the meantime, initially just as one chapter of Aeron Damphair on the Iron Islands at the Kingsmoot.
 * Robb Stark is slaughtered at the Red Wedding soon after, while Joffrey is poisoned at his own wedding to Margaery Tyrell.
 * The wedding is a trap, with Robb, his key supporters and most of his army massacred during the feast, a direct violation of ancient guest right customs, in what becomes known as the Red Wedding.
 * Robb is also slain at The Red Wedding along with his mother.
 * At the Red Wedding, Roose betr

[4] Define the prompt components and write the prompt construction function.

In [23]:
PROMPT_ROLE = "You are a knowledgeable Game of Thrones lore expert who answers questions about the world of Westeros.\n\n"

PROMPT_RULES = """\nRULES:
1. Answer the question using only the information provided in CONTEXT
2. Do not include information outside of CONTEXT
3. If the CONTEXT does not contain enough information to answer, say so clearly
4. Begin your response with "Answer: "
5. Write a concise, informative paragraph answer
"""

PROMPT_QUESTION = "\nQuestion: "

In [24]:
def prompt_qa(question, context_str):
    prompt = PROMPT_ROLE + context_str + PROMPT_RULES + PROMPT_QUESTION + question
    return prompt

[5] Write a function to call the LLM with the prompt.

In [25]:
def prompt_LLM(prompt):
    response: ChatResponse = chat(model='llama3.2', messages=[
      {
        'role': 'user',
        'content': prompt,
      },
    ])
    return response['message']['content']

[6] Write the full RAG pipeline function. It takes a query, retrieves sentences using hybrid search, builds the prompt, gets a response from the LLM, and prints the answer.

In [26]:
def ask(question, count=10, max_length=512, show_verbose=False):
    sentences = retrieve_sentences(question, count=count, max_length=max_length)
    context_str = create_context(sentences)
    prompt = prompt_qa(question, context_str)

    if show_verbose:
        print('--- Begin PROMPT ---')
        print(prompt)
        print('--- End PROMPT ---\n')

    answer = prompt_LLM(prompt)
    print(answer)

## Query Demonstrations

[7] Query 1: Who is Jon Snow and what is his true identity?

In [27]:
ask('Who is Jon Snow and what is his true identity?', show_verbose=True)

--- Begin PROMPT ---
You are a knowledgeable Game of Thrones lore expert who answers questions about the world of Westeros.

CONTEXT:
 * Unaware of Arya Stark's true identity, he takes her as his cupbearer and is impressed by her quick wit.
 * Sansa is called to give testimony, and although she reveals her true identity, she supports Baelish's story.
 * Reek returns with several hundred Bolton men, he kills Ser Rodrik Cassel and all the other northmen, but he then reveals his true identity as Roose Bolton's bastard Ramsay Snow.
 * Therefore, what the readers believe to be true may not necessarily be true.
 * There, he names a captive Arya Stark as his cupbearer, though it is left uncertain as to whether he knows her true identity.
 * Despite being encouraged to conceal his identity, Gendry reveals his parentage to Jon, and is allowed to join Jon in the journey beyond the Wall.
 * During their journey, Gendry discovers Arya's true identity and the two form a close friendship.
 * Jon Arr

[8] Query 2: What happened at the Red Wedding?

In [28]:
ask('What happened at the Red Wedding?', show_verbose=True)

--- Begin PROMPT ---
You are a knowledgeable Game of Thrones lore expert who answers questions about the world of Westeros.

CONTEXT:
 * She is married to Lord Edmure Tully as compensation at what becomes known as the Red Wedding.
 * However, just as they arrive, the Red Wedding happens and the Freys begin slaughtering the Starks.
 * Salon.com's Andrew Leonard "couldn't stop reading Martin because my desire to know what was going to happen combined with my absolute inability to guess what would happen and left me helpless before his sorcery.
 * A long prologue was to establish what had happened in the meantime, initially just as one chapter of Aeron Damphair on the Iron Islands at the Kingsmoot.
 * Robb Stark is slaughtered at the Red Wedding soon after, while Joffrey is poisoned at his own wedding to Margaery Tyrell.
 * The wedding is a trap, with Robb, his key supporters and most of his army massacred during the feast, a direct violation of ancient guest right customs, in what become

[9] Query 3: What is the history of House Targaryen and their dragons?

In [29]:
ask('What is the history of House Targaryen and their dragons?', show_verbose=True)

--- Begin PROMPT ---
You are a knowledgeable Game of Thrones lore expert who answers questions about the world of Westeros.

CONTEXT:
 * Their sigil is a three-headed black dragon on a red field, the reverse of House Targaryen.
 * Aided by their three formidable fire-breathing dragons, the Targaryen armies subdued six of the Seven Kingdoms through conquest or treaty, wiping out three of the seven ruling houses that refused to bend their knees, replacing house Durrandon with house Baratheon, house Gardener with house Tyrell, and house Hoare with houses Tully (in the Riverlands) and Greyjoy (on the Iron Islands).
 * Daenerys Targaryen
Daenerys Targaryen, referred to sometimes as 'Daenerys Stormborn', 'Khaleesi', the 'Mother of Dragons', is the daughter and youngest child of King Aerys II Targaryen and is one of the last surviving members of House Targaryen.
 * When the Doom came upon Valyria, House Targaryen survived along with the last of the Valyrian dragons.

RULES:
1. Answer the ques

[10] Query 4 (Failure case): A question the system cannot answer well because the topic is not in the database.

In [30]:
ask('What are the lyrics to the Game of Thrones theme song?', show_verbose=True)

--- Begin PROMPT ---
You are a knowledgeable Game of Thrones lore expert who answers questions about the world of Westeros.

CONTEXT:
 * Starting 2018, Diageo released several Game of Thrones themed whiskies.
 * Tourism Ireland has a Game of Thrones-themed marketing campaign similar to New Zealand's Tolkien-related advertising.
 * Themes
Wheareas modern fantasy often embraces magical elements, A Song of Ice and Fire series is generally praised for what is perceived as a sort of medieval realism.
 * It is his chance to escape the sordid and deadly "game of thrones," but he cannot bring himself to, confessing, "Bad people are what I'm good at."
 * The characters from the medieval fantasy television series Game of Thrones are adapted from George R. R. Martin’s novel series A Song of Ice and Fire.
 * The writers spent several weeks writing a character outline, including what material from the novels to use and the overarching themes.
 * Themes
Both television critics and historians have pr

[11] Queries 5 & 6: Comparing hybrid retrieval vs. term-only retrieval.

First, write a version of the retrieval function that uses only the match (IR) query, no KNN.

In [31]:
def retrieve_sentences_term_only(query, count=10, max_length=512):
    search_query = {
        'query': {
            'match': {
                'sentence': query
            }
        },
        'size': count
    }

    results_obj = client.search(index='got_lore', body=search_query)
    results = dict(results_obj)
    sentences = [hit['_source']['sentence'] for hit in results['hits']['hits']]
    truncated = [s[:max_length] for s in sentences]
    return truncated


def ask_term_only(question, count=10, max_length=512, show_verbose=False):
    sentences = retrieve_sentences_term_only(question, count=count, max_length=max_length)
    context_str = create_context(sentences)
    prompt = prompt_qa(question, context_str)

    if show_verbose:
        print('--- Begin PROMPT ---')
        print(prompt)
        print('--- End PROMPT ---\n')

    answer = prompt_LLM(prompt)
    print(answer)

Query 5: Ask about Daenerys Targaryen using the FULL hybrid pipeline.

In [32]:
print('=== HYBRID RETRIEVAL (match + KNN) ===')
ask('How did Daenerys Targaryen rise to power?', show_verbose=True)

=== HYBRID RETRIEVAL (match + KNN) ===
--- Begin PROMPT ---
You are a knowledgeable Game of Thrones lore expert who answers questions about the world of Westeros.

CONTEXT:
 * Tywin's death upsets the balance of power in King's Landing, namely by allowing the rise to power of the High Sparrow and the Faith Militant.
 * The Prince discusses the use of amoral ways and "how to do wrong" to gain power.
 * Her rise to power is aided by the historic birth of three dragons, hatched from eggs given to her as wedding gifts.
 * Tyrion arrives in Pentos, where Varys reveals that he has been conspiring to restore House Targaryen to power, and asks Tyrion to journey with him to meet Daenerys Targaryen in Meereen.
 * Petyr helps Eddard expose the secret parentage of the royal children, but advises him to abet Joffrey's rise to power in order to consolidate their own.
 * Later, she conquers Yunkai and Meereen, the latter Daenerys settles in to learn how to rule.
 * After Daenerys conquers the city sh

Query 6: Same question using TERM-ONLY retrieval (no embeddings).

In [33]:
print('=== TERM-ONLY RETRIEVAL (match query only) ===')
ask_term_only('How did Daenerys Targaryen rise to power?', show_verbose=True)

=== TERM-ONLY RETRIEVAL (match query only) ===
--- Begin PROMPT ---
You are a knowledgeable Game of Thrones lore expert who answers questions about the world of Westeros.

CONTEXT:
 * Tywin's death upsets the balance of power in King's Landing, namely by allowing the rise to power of the High Sparrow and the Faith Militant.
 * The Prince discusses the use of amoral ways and "how to do wrong" to gain power.
 * Her rise to power is aided by the historic birth of three dragons, hatched from eggs given to her as wedding gifts.
 * Tyrion arrives in Pentos, where Varys reveals that he has been conspiring to restore House Targaryen to power, and asks Tyrion to journey with him to meet Daenerys Targaryen in Meereen.
 * Petyr helps Eddard expose the secret parentage of the royal children, but advises him to abet Joffrey's rise to power in order to consolidate their own.
 * Later, she conquers Yunkai and Meereen, the latter Daenerys settles in to learn how to rule.
 * After Daenerys conquers the

[12] Additional query: Who are the members of the Night's Watch and what is their purpose?

In [34]:
ask("Who are the members of the Night's Watch and what is their purpose?", show_verbose=True)

--- Begin PROMPT ---
You are a knowledgeable Game of Thrones lore expert who answers questions about the world of Westeros.

CONTEXT:
 * Mance Rayder
Mance Rayder is a former member of the Night's Watch who later deserted.
 * As the series premiered, TV Guide called Harington a "soulful heartthrob" whose Jon is idolized by his younger siblings and who "seeks purpose" by joining the Night's Watch.
 * The North
Night's Watch
The Night's Watch is a sworn brotherhood of men who patrol the Wall.
 * As their only ally beyond the Wall, Commander Mormont and the Night's Watch are forced to endure his insults and outrageous demands.
 * Waymar is killed by a White Walker
Gared (portrayed by Dermot Keaney) A member of the Night's Watch.
 * Night's Watch
Will (portrayed by Bronson Webb) A ranger of the Night's Watch, who alongside Ser Waymar Royce and Gared investigate reports of wildlings in the Haunted Forest.

RULES:
1. Answer the question using only the information provided in CONTEXT
2. Do no